# Interactive Exploration of Hollow Performance

This notebook allows you to step through the mathematical model and simulations.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.hollow_performance import HollowPerformanceModel

## 1. Simple Risk Boundary

In [ ]:
model = HollowPerformanceModel()
for kappa in [1, 3, 5]:
    print(f"κ={kappa}: safe D(1-V) < {model.risk_boundary(kappa):.3f}")

## 2. Learning‑Loop Strength Function

In [ ]:
p = np.linspace(0,1,100)
G = model.learning_loop_strength(p, p, p, p)
plt.plot(p, G)
plt.xlabel('P=F=J=B')
plt.ylabel('G (learning-loop strength)')
plt.title('Softened product G = α·P·F·J·B + β')
plt.grid(True)
plt.show()

## 3. Custom Simulation

In [ ]:
def simulate_custom(blind_trust, A=0.9, T=200):
    K = 0.8
    P = F = J = B = 1.0 - blind_trust
    Q = model.context_quality(1,1,1)
    G = model.learning_loop_strength(P, F, J, B)
    ret = model.retention_factor(0)
    K_hist, Y_hist = [], []
    for _ in range(T):
        L = model.durable_learning(K, Q, G, ret)
        off = model.offload_loss(A, P, B)
        forg = model.forget_loss(K)
        K = model.mastery_update(K, L, off, forg)
        Y = model.observed_performance(K, A)
        K_hist.append(K); Y_hist.append(Y)
    return K_hist, Y_hist

bt_values = [0, 0.1, 0.2, 0.3, 0.4]
plt.figure(figsize=(12,6))
for bt in bt_values:
    K, Y = simulate_custom(bt, T=150)
    plt.plot(K, label=f'K (bt={bt})')
    plt.plot(Y, '--', label=f'Y (bt={bt})')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.title('Effect of blind trust on mastery and performance')
plt.grid(True)
plt.show()

## 4. Hollow Performance (Degrading Learning Loop)

In [ ]:
def hollowing_sim(steps=200, hollow_rate=0.01):
    K = 0.8
    hist_K, hist_Y = [], []
    for t in range(steps):
        baseline = max(0.9 - hollow_rate * t, 0.1)
        P = baseline * np.random.beta(2,2)
        F = baseline * np.random.beta(2,2)
        J = baseline * np.random.beta(2,2)
        B = baseline * np.random.beta(2,2)
        Q = model.context_quality(baseline, baseline, baseline)
        G = model.learning_loop_strength(P, F, J, B)
        ret = model.retention_factor(baseline * np.random.beta(2,5))
        L = model.durable_learning(K, Q, G, ret)
        off = model.offload_loss(0.9, P, B)
        forg = model.forget_loss(K)
        K = model.mastery_update(K, L, off, forg)
        Y = model.observed_performance(K, 0.9)
        hist_K.append(K); hist_Y.append(Y)
    return hist_K, hist_Y

K_h, Y_h = hollowing_sim(300, 0.008)
plt.plot(K_h, label='Mastery')
plt.plot(Y_h, '--', label='Performance')
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Hollow performance: Y high, K collapses')
plt.legend()
plt.grid(True)
plt.show()
print(f"Final: K={K_h[-1]:.3f}, Y={Y_h[-1]:.3f}, gap={Y_h[-1]-K_h[-1]:.3f}")